In [4]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from accelerate import Accelerator, notebook_launcher

In [14]:
def train():
    accelerator = Accelerator(mixed_precision="fp16")

    torch.manual_seed(42)

    X = torch.randn(6000, 20)
    y = X.sum(dim=1, keepdim=True) + 0.1 * torch.randn(6000, 1)

    dataloader = DataLoader(
        TensorDataset(X, y),
        batch_size=64,
        shuffle=True
    )

    model = nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 1)
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)

    model, optimizer, dataloader = accelerator.prepare(
        model, optimizer, dataloader
    )

    print(
        f"GPU {accelerator.process_index} | "
        f"Batches: {len(dataloader)} | "
        f"Device: {accelerator.device}",
        flush=True
    )

    for epoch in range(50):
        model.train()

        total_loss = torch.zeros((), device=accelerator.device)
        num_batches = torch.zeros((), device=accelerator.device)

        for inputs, targets in dataloader:
            optimizer.zero_grad()

            with accelerator.autocast():
                predictions = model(inputs)
                loss = nn.functional.mse_loss(predictions, targets)

            accelerator.backward(loss)
            optimizer.step()

            total_loss += loss.detach().float()
            num_batches += 1

        # Aggregate across all 6 GPUs
        global_loss = accelerator.reduce(total_loss, reduction="sum")
        global_batches = accelerator.reduce(num_batches, reduction="sum")

        if accelerator.is_main_process:
            if global_batches.item() == 0:
                raise RuntimeError(
                    "DataLoader produced zero batches. "
                    "Check dataset size, batch size, and drop_last."
                )

            avg_loss = (global_loss / global_batches).item()
            print(f"Epoch {epoch + 1}/5 | Loss: {avg_loss:.6f}", flush=True)

        accelerator.wait_for_everyone()

In [15]:
# Cell 3 — Launch training on all six RTX 2080 Ti GPUs
notebook_launcher(
    train,
    num_processes=6,
    mixed_precision="fp16",
)

Launching training on 6 CUDAs.
GPU 3 | Batches: 16 | Device: cuda:3GPU 1 | Batches: 16 | Device: cuda:1GPU 5 | Batches: 16 | Device: cuda:5
GPU 2 | Batches: 16 | Device: cuda:2

GPU 4 | Batches: 16 | Device: cuda:4

GPU 0 | Batches: 16 | Device: cuda:0
Epoch 1/5 | Loss: 12.848180
Epoch 2/5 | Loss: 1.006967
Epoch 3/5 | Loss: 0.273361
Epoch 4/5 | Loss: 0.098449
Epoch 5/5 | Loss: 0.058881
Epoch 6/5 | Loss: 0.043744
Epoch 7/5 | Loss: 0.037435
Epoch 8/5 | Loss: 0.032806
Epoch 9/5 | Loss: 0.029536
Epoch 10/5 | Loss: 0.026114
Epoch 11/5 | Loss: 0.023495
Epoch 12/5 | Loss: 0.021268
Epoch 13/5 | Loss: 0.019658
Epoch 14/5 | Loss: 0.018102
Epoch 15/5 | Loss: 0.017017
Epoch 16/5 | Loss: 0.016132
Epoch 17/5 | Loss: 0.015486
Epoch 18/5 | Loss: 0.014887
Epoch 19/5 | Loss: 0.014465
Epoch 20/5 | Loss: 0.014109
Epoch 21/5 | Loss: 0.013815
Epoch 22/5 | Loss: 0.013590
Epoch 23/5 | Loss: 0.013343
Epoch 24/5 | Loss: 0.013008
Epoch 25/5 | Loss: 0.012673
Epoch 26/5 | Loss: 0.012723
Epoch 27/5 | Loss: 0.012488